# Using MCP Tools with Azure AI Foundry Agents

This notebook demonstrates how to use Model Context Protocol (MCP) tools with Azure AI Foundry agents. Azure AI Foundry provides seamless integration with hosted MCP servers, eliminating infrastructure management while providing secure, controlled access to external tools.

## What You'll Learn:
- Basic hosted MCP tool integration with Azure AI Foundry agents
- Multi-tool MCP configuration with different approval modes
- Azure AI observability for monitoring and tracing
- Microsoft Learn MCP server integration for documentation queries

## Key Features:
- **Hosted MCP Server**: Managed by Azure AI Foundry, no infrastructure to maintain
- **Persistent Agents**: Server-side agent creation with stateful conversations
- **Tool Approval Workflow**: Configurable approval mechanisms for MCP tool invocations
- **Observability**: Built-in monitoring and tracing for production scenarios

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Project**: Access to an Azure AI Foundry project with deployed models
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with connection details
4. **Dependencies**: Required agent-framework packages installed

If you need to use a different tenant, specify the tenant ID:
```bash
az login --tenant <tenant-id>
```

## Import Libraries

Import the required libraries for Azure AI agent functionality with MCP integration.

In [ ]:
import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv

from agent_framework import HostedMCPTool
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

# Load environment variables from .env file
load_dotenv('../../.env')

# Verify required environment variables
print("Checking environment variables...")
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")
petstore_mcp_url = os.getenv("CUSTOM_PETSTORE_MCP_URL")

if endpoint:
    print(f"✅ AZURE_AI_PROJECT_ENDPOINT: {endpoint[:50]}...")
    print(f"✅ AZURE_AI_MODEL_DEPLOYMENT_NAME: {model}")
else:
    print("❌ AZURE_AI_PROJECT_ENDPOINT is not set!")
    print("Please configure your .env file with the required variables.")

## Example 1: Basic MCP Integration

This example demonstrates the simplest way to create an Azure AI Foundry agent with a hosted MCP tool. The agent can search Microsoft Learn documentation to answer questions.

In [ ]:
async def basic_foundry_mcp_example():
    """Basic example of Azure AI Foundry agent with hosted MCP tools."""
    async with AzureCliCredential() as credential:
        async with AzureAIAgentClient(credential=credential) as chat_client:
            # Enable Azure AI observability (optional but recommended)
            #print("🔍 Setting up Azure AI observability...")
            #await chat_client.setup_azure_ai_observability()
            print("✅ Azure AI observability enabled\n")

            # Create agent with hosted MCP tool
            print("🤖 Creating MicrosoftLearnAgent with hosted MCP tool...")
            agent = chat_client.create_agent(
                name="MicrosoftLearnAgent", 
                instructions="You answer questions by searching Microsoft Learn content only.",
                tools=HostedMCPTool(
                    name="Microsoft Learn MCP",
                    url="https://learn.microsoft.com/api/mcp",
                    approval_mode="never_require",  # Auto-approve tool calls
                ),
            )
            print(f"✅ Created agent: {agent.name}\n")

            # Simple query without approval workflow
            query = "Please summarize the Azure AI Agent documentation related to MCP tool calling?"
            print(f"🤔 Query: {query}\n")
            
            result = await agent.run(query)
            
            print(f"🤖 Response:\n{result.text}")
            print(f"\n✅ Basic example completed!")

# Run the basic example (use await in notebooks, asyncio.run() in scripts)
await basic_foundry_mcp_example()

## Example 2: Multi-Tool MCP Configuration with Custom Headers

This example demonstrates using multiple hosted MCP tools with custom headers for authentication. This is useful when you want to integrate multiple external services with varying authentication requirements.

### Setting Up Your Own MCP Server

To use this example, you can either:

1. **Create your own MCP server** using Azure API Management (APIM):
   - Follow the [APIM MCP documentation](https://learn.microsoft.com/en-us/azure/api-management/export-rest-mcp-server) to expose any REST API as an MCP server
   - **Important:** Disable diagnostic response body logging in APIM to prevent interference with MCP streaming
   - Update the URL in the code below to match your APIM MCP endpoint

2. **Use a public MCP server** that requires authentication:
   - Many MCP servers require API tokens passed via headers
   - Store tokens securely in your `.env` file (never commit to version control)
   - Use the authentication pattern: `headers={"Authorization": f"Bearer {api_token}"}`

### Example: Azure APIM-Hosted Petstore MCP

This example uses a custom MCP server hosted through Azure API Management, exposing the Swagger Petstore API with 8 tools:
- `findPetById` - Get pet information by ID
- `findsPetsByStatus` - Search pets by status (available, pending, sold)
- `findsPetsByTags` - Search pets by tags
- `returnsPetInventoriesByStatus` - Get inventory counts
- `getUserByUserName` - Get user information
- And more...

**Note:** Replace the URL below with your own APIM MCP endpoint or another hosted MCP server.

In [ ]:
async def multi_tool_mcp_example():
    """Example using multiple hosted MCP tools with custom headers."""
    
    # Get API key from environment (if your MCP server requires it)
    api_key = os.getenv("MCP_API_KEY")
    
    async with AzureCliCredential() as credential:
        async with AzureAIAgentClient(credential=credential) as chat_client:
            # Build tools list with multiple MCP servers
            petstore_headers = {"api-key": api_key} if api_key else {}
            
            # Debug: Show headers being sent
            print(f"🔍 Petstore MCP Configuration:")
            print(f"   URL: {petstore_mcp_url}")
            print(f"   Headers: {petstore_headers}")
            print()
            
            tools = [
                HostedMCPTool(
                    name="Microsoft_Learn_MCP",
                    url="https://learn.microsoft.com/api/mcp",
                    approval_mode="never_require",
                ),
                HostedMCPTool(
                    name="Petstore_MCP_APIM",
                    url=petstore_mcp_url,
                    approval_mode="never_require",
                    headers=petstore_headers,
                ),
            ]
            
            print("✅ Configured MCP tools:")
            print("   • Microsoft Learn MCP (documentation) - auto-approved")
            print("   • Petstore MCP via Azure APIM (8 REST API tools) - requires approval")

            # Create agent with multiple MCP tools
            print(f"🤖 Creating MultiToolAgent with {len(tools)} MCP tool(s)...")
            agent = chat_client.create_agent(
                name="MultiToolAgent",
                instructions="You can search documentation and access integrations through MCP tools.",
                tools=tools,
            )
            print(f"✅ Created agent: {agent.name}\n")

            # Query that will use the MCP tools
            query = "What are the key features of Azure AI Foundry? I would also like to know about pets that have been sold"
            print(f"🤔 Query: {query}\n")
            
            try:
                result = await agent.run(query)
                print(f"🤖 Response:\n{result.text}")
                print(f"\n✅ Multi-tool example completed!")
            except Exception as e:
                print(f"❌ Error occurred: {type(e).__name__}")
                print(f"   Message: {str(e)}")
                import traceback
                print(f"\n📋 Full traceback:")
                traceback.print_exc()

# Run the multi-tool example
await multi_tool_mcp_example()

## Example 3: Thread-Based Conversation with MCP Tools

This example demonstrates how to use threads to maintain conversation context across multiple queries while using hosted MCP tools.

In [ ]:
async def thread_based_mcp_example():
    """Example showing thread-based conversation with hosted MCP tools."""
    async with AzureCliCredential() as credential:
        async with AzureAIAgentClient(credential=credential) as chat_client:
            print("🔍 Setting up Azure AI observability...")
            #await chat_client.setup_azure_ai_observability()
            print("✅ Azure AI observability enabled\n")

            # Create agent with hosted MCP tool
            print("🤖 Creating DocsAgent with Microsoft Learn MCP...")
            agent = chat_client.create_agent(
                name="DocsAgent",
                instructions="You are a helpful assistant that can help with Microsoft documentation questions.",
                tools=HostedMCPTool(
                    name="Microsoft Learn MCP",
                    url="https://learn.microsoft.com/api/mcp",
                    approval_mode="never_require",  # Auto-approve tool calls
                ),
            )
            print(f"✅ Created agent: {agent.name}\n")
            
            # Create a new thread for conversation
            thread = agent.get_new_thread()
            print(f"📝 Created new conversation thread\n")
            
            # First query
            query1 = "How to create an Azure storage account using az cli?"
            print(f"=== Query 1 ===")
            print(f"🤔 User: {query1}\n")
            result1 = await agent.run(query1, thread=thread, store=True)
            print(f"🤖 {agent.name}: {result1.text}")
            
            print("\n" + "="*60 + "\n")
            
            # Second query (uses same thread for context)
            query2 = "What is Microsoft Agent Framework?"
            print(f"=== Query 2 ===")
            print(f"🤔 User: {query2}\n")
            result2 = await agent.run(query2, thread=thread, store=True)
            print(f"🤖 {agent.name}: {result2.text}")
            
            print(f"\n✅ Thread-based conversation completed!")

# Run the thread-based example
await thread_based_mcp_example()

## Example 4: Tool Comparison Test (get_weather vs HostedMCPTool)

This cell allows you to test different tool combinations to diagnose the second-query failure issue:
- **Test 1**: Only get_weather function tool
- **Test 2**: Only HostedMCPTool (Microsoft Learn MCP)
- **Test 3**: Both tools together

Toggle the tools list to see which configuration works and which fails.

In [ ]:
async def tool_comparison_test():
    """Flexible test to compare different tool configurations."""
    import asyncio
    from typing import Annotated
    from pydantic import Field
    
    # Define get_weather function tool
    def get_weather(
        location: Annotated[str, Field(description="The location to get the weather for.")],
    ) -> str:
        """Get the weather for a given location."""
        conditions = ["sunny", "cloudy", "rainy", "stormy"]
        temperature = 22
        return f"The weather in {location} is {conditions[0]} with a high of {temperature}°C."
    
    async with AzureCliCredential() as credential:
        async with AzureAIAgentClient(credential=credential) as chat_client:
            # ============================================
            # CONFIGURE TOOLS HERE - Comment/uncomment to test different combinations
            # ============================================
            
            # Test 1: Only function tool (should work)
            # tools = [get_weather]
            # test_name = "get_weather only"
            # query1 = "What's the weather in New York?"
            # query2 = "What about London?"
            
            # Test 2: Only MCP tool (likely fails on second query)
            api_key = os.getenv("MCP_API_KEY")
            petstore_headers = {"api-key": api_key} if api_key else {}
            tools = [
                HostedMCPTool(
                    name="Petstore_MCP_APIM",
                    url=petstore_mcp_url,
                    approval_mode="never_require",
                    headers=petstore_headers,
                )
            ]
            # tools = [
            #     HostedMCPTool(
            #         name="Microsoft_Learn_MCP",
            #         url="https://learn.microsoft.com/api/mcp",
            #         approval_mode="never_require",
            #     )
            # ]
            test_name = "HostedMCPTool only"
            query1 = "How to create Azure storage account?"
            query2 = "What is Azure AI Foundry?"
            
            # Test 3: Both tools together
            # tools = [
            #     get_weather,
            #     HostedMCPTool(
            #         name="Microsoft_Learn_MCP",
            #         url="https://learn.microsoft.com/api/mcp",
            #         approval_mode="never_require",
            #     )
            # ]
            # test_name = "Both get_weather AND HostedMCPTool"
            # query1 = "How to create an Azure storage account?"
            # query2 = "What's the weather in Paris?"
            
            # ============================================
            # TEST EXECUTION
            # ============================================
            
            print(f"🧪 TEST: {test_name}")
            print(f"📋 Tools configured: {len(tools)} tool(s)")
            print("="*70 + "\n")
            
            agent = chat_client.create_agent(
                name="TestAgent",
                instructions="You are a helpful assistant.",
                tools=tools,
            )
            print(f"✅ Created agent: {agent.name}\n")
            
            # First query
            print(f"=== Query 1 ===")
            print(f"🤔 User: {query1}\n")
            try:
                result1 = await agent.run(query1)
                print(f"✅ Query 1 SUCCEEDED")
                print(f"🤖 Response: {result1.text}\n")
            except Exception as e:
                print(f"❌ Query 1 FAILED: {e}\n")
                return
            
            print("="*70 + "\n")
            
            # Small delay
            print("⏳ Waiting 2 seconds...")
            await asyncio.sleep(2)
            print()
            
            # Second query
            print(f"=== Query 2 ===")
            print(f"🤔 User: {query2}\n")
            try:
                result2 = await agent.run(query2)
                print(f"✅ Query 2 SUCCEEDED")
                print(f"🤖 Response: {result2.text}\n")
                print("="*70)
                print(f"🎉 SUCCESS: Both queries worked with {test_name}!")
                print("="*70)
            except Exception as e:
                print(f"❌ Query 2 FAILED")
                print(f"   Error: {type(e).__name__}: {str(e)}")
                print("="*70)
                print(f"⚠️  FAILURE: Second query failed with {test_name}")
                print("="*70)

# Run the test
await tool_comparison_test()

## Key Takeaways

1. **Hosted MCP Tools**: Enable integration with external Model Context Protocol servers
2. **User Approval Workflows**: Provide security by requiring consent for function calls
3. **Thread Management**: Maintain conversation context across multiple queries
4. **Azure AI Observability**: Built-in monitoring and tracing for agent interactions
5. **Microsoft Learn Integration**: Access to comprehensive Microsoft documentation
6. **Error Handling**: Robust error handling for production scenarios

## Best Practices

1. **Security First**: Always implement proper approval workflows for function calls
2. **Observability**: Enable Azure AI observability for monitoring and debugging
3. **Thread Management**: Use threads to maintain conversation context
4. **Error Handling**: Implement comprehensive error handling for reliability
5. **Custom Approvals**: Tailor approval logic to your specific security requirements
6. **Resource Cleanup**: Properly manage agent and thread lifecycles

## Use Cases

- **Documentation Assistance**: AI-powered help with Microsoft technologies
- **Technical Support**: Automated support with human oversight
- **Knowledge Management**: Organizational knowledge base integration
- **Training and Education**: Interactive learning with documentation
- **Code Generation**: Context-aware code examples and templates
- **Compliance**: Secure function execution with approval workflows